# Where does a well's water come from?

A water utility pumping from a valley aquifer has neighbors: a lake at the north
end, rain falling on the whole valley, and the water that was already underground
when pumping started. A regulator asking whether the wells are drawing lake
water, and a manager asking how long the aquifer's own storage will last, are
asking the same question in two ways. Heads cannot answer it, because every drop
of water in the aquifer has the same head whatever its origin.

## What this notebook covers

Label each source of water with a tracer and let the transport model carry the
labels to the wells. The concentration that arrives is the fraction of the well's
water that came from that source.

By the end of this notebook you will be able to:

- set up an **end-member** simulation, where one source is given a concentration
  of 1 and everything else 0,
- tell a source apart from an initial condition, which needs a different kind of
  run,
- read source fractions from the water the wells pump, and
- check the result by adding the fractions up.

The model is the Voronoi-grid synthetic valley that ships with this repository,
`synthetic-valley-vg-advanced-annual`, the same one
[`flopy-intro-gwt-a`](flopy-intro-gwt-a.ipynb) and
[`flopy-intro-gwt-b`](flopy-intro-gwt-b.ipynb) use. Two multi-aquifer wells,
`reilly` in the northeast of the valley and `vc` in the southwest, pump from the
lower aquifer (layers 4 and 5). The model is in meters and days.

Import the packages this notebook uses. `mf6_gwt_source_water` holds the model
builders, so the cells below stay about the water rather than about flopy.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import mf6_gwt_source_water as sw
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries, label_map_axes

_, mf6_exe = find_mf6_libraries()

## The idea: label the water

A tracer that does not decay, sorb, or change the flow is a label. Put a
concentration of 1 on one source and 0 on the others, and every water molecule
carries a number saying whether it came from that source. Mix the water in an
aquifer and the concentration at any point becomes the fraction of the water at
that point that started at the labeled source.

That gives one transport simulation per source. Here there are three sources,
so three runs:

| end member | how it is labeled |
|---|---|
| lake | water leaking from the lake into the aquifer has a concentration of 1 |
| recharge | water reaching the water table from the unsaturated zone has a concentration of 1 |
| initial | the aquifer starts at a concentration of 1 everywhere |

Recharge here is what is left of the rain soaking into the land after
evapotranspiration, handed to the aquifer by the unsaturated-zone flow (UZF)
package. The third end member is different in kind. Water already in the
aquifer is not a source pouring in at a boundary; it is what the model starts
with, so it is labeled through the initial condition instead.

One flow run serves all three. A source concentration is an auxiliary variable
on a flow package: an extra number that MODFLOW does not use for flow but writes
to the budget file with every flow of that package. The flow model gives both
the lake and the unsaturated zone a concentration of 1. Each transport run then
picks the label it reads: its Source and Sink Mixing (SSM) package names the one
package whose water is traced, and water from every package it does not name
enters the aquifer at a concentration of 0. The stream and every other boundary
only take water out of the aquifer in this run, so they need no label.

In [ ]:
for name, (labeled, initial) in sw.END_MEMBERS.items():
    traced = ", ".join(labeled) if labeled else "none"
    print(
        f"{name:9s} labeled package: {traced:6s}   initial concentration {initial:.0f}"
    )
print()
print(f"wells: {', '.join(sw.WELL_NAMES)}")

## Run the three end members

Run the flow model once, then the three transport models that read it. The flow
model starts from a steady period with the wells off, then pumps for 20 one-year
periods. The steady period is shortened to one day, which does not change a
steady flow solution, so the water in the aquifer at the start of the transport
runs is the water there when pumping starts. The transport models take the same
steps as the flow model, because the Flow Model Interface reads one flow budget
record per transport time step.

Each well draws from two layers, and the water from the two screens mixes in the
well bore. A multi-aquifer well transport (MWT) package follows that mixing, so
the concentration it reports is the concentration of the water the well pumps.

Four simulations, about a minute and a half.

In [ ]:
model_root = pl.Path("models/gwt-source-water")
fractions = sw.source_fractions(model_root, mf6_exe)

# years since pumping started, at the end of each one-year period
years = (fractions["lake"]["time"] - sw.SPINUP) / 365.25
print(f"output times: {len(years)}, ending at {years.iloc[-1]:.0f} years")

## What the wells produce

The concentration at each well is that end member's share of the water it
pumps. Read the three runs at the end of the simulation.

In [ ]:
final = sw.final_fractions(fractions)
final["total"] = final.sum(axis=1)
print(final.to_string(float_format=lambda v: f"{v:8.4f}"))

The totals are the check worth making. The three sources are the only water the
wells can draw, so the fractions have to add to 1 at every well and at every
time. Anything else means a source is missing, or a concentration was set on the
wrong package.

In [ ]:
print(f"largest departure from 1: {sw.fraction_closure(fractions):.2e}")

Plot the three fractions through time at each well, stacked so the total shows.

In [ ]:
colors = {"lake": "tab:blue", "recharge": "tab:green", "initial": "tab:brown"}

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [list("AB")], figsize=(7.0, 3.6), layout="constrained", sharey=True
    )
    for letter, well in zip("AB", sw.WELL_NAMES):
        ax = axd[letter]
        ax.stackplot(
            years,
            [fractions[member][well] for member in sw.END_MEMBERS],
            labels=list(sw.END_MEMBERS),
            colors=[colors[member] for member in sw.END_MEMBERS],
        )
        ax.set_xlim(0.0, years.iloc[-1])
        ax.set_ylim(0.0, 1.0)
        ax.set_xlabel("time since pumping started, in years")
        ax.set_title(f"{letter}. Well {well}")
    axd["A"].set_ylabel("fraction of the water pumped")
    axd["B"].legend(loc="lower right", fontsize=7)

**What to look for.** Each band is one source, and the three fill the panel
because they are the only water there is. Both wells start out pumping the water
already in the aquifer (brown) and turn steadily to recharge (green): after 10
years recharge supplies 0.27 of what reilly produces and 0.33 of what vc
produces, and after 20 years 0.54 at both. The water that was in the aquifer
when pumping started is still 0.46 of reilly's water and 0.45 of vc's at 20
years.

The blue band is a sliver at vc and missing at reilly. Lake water reaches vc,
2.4 km south of the lake, after about 10 years, at 0.0005 of its water, and makes
up 0.013 by 20 years. At reilly the lake fraction never exceeds 6e-6, which is
nothing, although reilly is only 560 m east of the lake: before pumping the
groundwater there moves west toward the lake, so the well sits upstream of it.

## Where the labeled water goes

Reading the wells says how much lake water arrives but not where it goes. Map
two end members in layer 4, the upper of the two layers the wells are screened
in: the lake water, and the recharge.

In [ ]:
flow_sim = flopy.mf6.MFSimulation.load(
    sim_ws=str(model_root / "flow"),
    verbosity_level=0,
    load_only=["disv", "maw"],
)
gwf = flow_sim.get_model()
wells = gwf.get_package("pwell")
# the cell at the top of each well's screen, by the well's name
well_cells = {
    str(well["boundname"]): tuple(connection["cellid"])
    for well in wells.packagedata.array
    for connection in wells.connectiondata.array
    if connection["ifno"] == well["ifno"] and connection["icon"] == 0
}
map_layer = min(cellid[0] for cellid in well_cells.values())


def end_member_map(end_member):
    """Concentration in the mapped layer at the end of the run."""
    path = model_root / f"{end_member}-transport" / f"{sw.GWT_NAME}.ucn"
    conc = flopy.utils.HeadFile(path, text="CONCENTRATION").get_data()
    # one value per cell in the layer, with MODFLOW's placeholder for the
    # inactive cells taken out
    layer = conc[map_layer].ravel()
    return np.where(layer > 1.0e29, np.nan, layer)


with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(9.0, 6.5), layout="constrained"
    )
    for letter, end_member in zip("AB", ("lake", "recharge")):
        ax = axd[letter]
        values = end_member_map(end_member)
        pmv = flopy.plot.PlotMapView(gwf, ax=ax, layer=map_layer)
        patches = pmv.plot_array(values, cmap="viridis", vmin=0.0, vmax=1.0)
        pmv.plot_grid(linewidth=0.15, color="0.9")
        for well, cellid in well_cells.items():
            x = gwf.modelgrid.xcellcenters[cellid[1]]
            y = gwf.modelgrid.ycellcenters[cellid[1]]
            ax.plot(x, y, "o", color="red", ms=5)
            ax.annotate(
                well,
                (x, y),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
                bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "none"},
            )
        label_map_axes(ax)
        ax.set_title(f"{letter}. {end_member.capitalize()} water")
    fig.colorbar(
        patches,
        ax=list(axd.values()),
        shrink=0.5,
        label=f"fraction of the water in layer {map_layer + 1}",
    )

**What to look for.** Panel A shows lake water spread through layer 4 south of
the lake, up to 0.57 of the water at its strongest, and moving down the western
side of the valley toward vc, where it is still a small share. None reaches the
eastern side of the valley or reilly. Panel B is close to the opposite: recharge
has replaced up to 0.85 of the water in the east and south, but little beneath
the lake, where no rain soaks through the land and the water is lake water or
water that was there when pumping started.

The two panels together explain the stacked bands. The wells sit where recharge
fills the aquifer, and in 20 years lake water has only begun to reach vc.

## Recap

- Labeling a source with a tracer concentration of 1 turns a transport model
  into a way of measuring where a well's water comes from, which heads alone
  cannot say.
- Each source needs its own transport simulation. One flow run can carry the
  labels of every source, and each transport run picks the one it reads through
  SSM; water already in the aquifer is labeled through the initial condition
  rather than through a boundary.
- The fractions at a well have to add to 1, and checking that is how you know
  the set of sources is complete. Here they close to within 2e-4.
- After 20 years of pumping, these wells produce 0.54 recharge and 0.45 to 0.46
  water that was already in the aquifer; only vc pumps any lake water,
  0.013 of its water.